# SBV2 00 - Environment And Register

Purpose: confirm that the local Sandbox V2 environment is ready before data download and extraction work begins.

This notebook is deliberately defensive. Missing optional packages or gated model checkpoints should appear as status rows, not hard failures.

## Notebook Contract

Expected outputs:

- hardware and runtime summary
- core package audit
- local model inventory
- source/model register preview
- audit CSV files in `outputs/sandbox_v2/audit_tables/`

Nothing in this notebook downloads data or loads heavyweight models.

In [1]:
from __future__ import annotations

import importlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "sandbox_v2").exists():
    candidates = [p for p in ROOT.parents if (p / "sandbox_v2").exists()]
    if candidates:
        ROOT = candidates[0]

os.chdir(ROOT)

CACHE_DIR = Path(os.environ.get("MENUFORGE_CACHE_HOME", ROOT / ".cache")).resolve()
os.environ["XDG_CACHE_HOME"] = str(CACHE_DIR)
os.environ["PADDLE_PDX_CACHE_HOME"] = str(CACHE_DIR / "paddlex")
os.environ["MPLCONFIGDIR"] = str(CACHE_DIR / "matplotlib")
os.environ["PIP_CACHE_DIR"] = str(CACHE_DIR / "pip")
os.environ["HF_HOME"] = str(CACHE_DIR / "huggingface")
os.environ["HF_HUB_CACHE"] = str(Path(os.environ["HF_HOME"]) / "hub")
os.environ["HF_XET_CACHE"] = str(Path(os.environ["HF_HOME"]) / "xet")
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
os.environ["IPYTHONDIR"] = str(CACHE_DIR / "ipython")
os.environ["JUPYTER_CONFIG_DIR"] = str(CACHE_DIR / "jupyter" / "config")
os.environ["JUPYTER_DATA_DIR"] = str(CACHE_DIR / "jupyter" / "data")
os.environ["JUPYTER_RUNTIME_DIR"] = str(CACHE_DIR / "jupyter" / "runtime")

for key in ["PADDLE_PDX_CACHE_HOME", "MPLCONFIGDIR", "PIP_CACHE_DIR", "HF_HOME", "HF_HUB_CACHE", "HF_XET_CACHE", "TORCH_HOME", "IPYTHONDIR", "JUPYTER_CONFIG_DIR", "JUPYTER_DATA_DIR", "JUPYTER_RUNTIME_DIR"]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

SANDBOX_DIR = ROOT / "sandbox_v2"
REGISTER_PATH = SANDBOX_DIR / "SOURCE_AND_MODEL_REGISTER.md"
AUDIT_DIR = ROOT / "outputs" / "sandbox_v2" / "audit_tables"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Register: {REGISTER_PATH}")
print(f"Audit output dir: {AUDIT_DIR}")
print(f"Project cache dir: {CACHE_DIR}")

Project root: /home/endy/menuforge
Register: /home/endy/menuforge/sandbox_v2/SOURCE_AND_MODEL_REGISTER.md
Audit output dir: /home/endy/menuforge/outputs/sandbox_v2/audit_tables
Project cache dir: /home/endy/menuforge/.cache


In [2]:
def run_cmd(cmd: list[str], timeout: int = 30) -> tuple[int, str, str]:
    try:
        proc = subprocess.run(
            cmd,
            cwd=ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
            check=False,
        )
        return proc.returncode, proc.stdout.strip(), proc.stderr.strip()
    except Exception as exc:
        return -1, "", repr(exc)


def package_version(dist_name: str) -> str | None:
    try:
        return importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def bytes_human(n_bytes: int | float | None) -> str:
    if n_bytes is None:
        return "n/a"
    value = float(n_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(value) < 1024 or unit == "TB":
            return f"{value:.1f} {unit}"
        value /= 1024
    return f"{value:.1f} TB"


def du_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    code, out, _ = run_cmd(["du", "-sb", str(path)], timeout=60)
    if code == 0 and out:
        return int(out.split()[0])
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def count_files(path: Path, pattern: str = "*") -> int:
    if not path.exists() or not path.is_dir():
        return 0
    return sum(1 for p in path.rglob(pattern) if p.is_file())


def first_existing(path: Path, names: list[str]) -> str:
    if path.is_file():
        return path.name
    for name in names:
        if (path / name).exists():
            return name
    return ""


print("Helper functions ready.")

Helper functions ready.


## Runtime And Hardware

In [3]:
git_code, git_branch, _ = run_cmd(["git", "branch", "--show-current"])
git_status_code, git_status, _ = run_cmd(["git", "status", "--short"])

runtime_rows = [
    {"item": "timestamp_local", "value": datetime.now().isoformat(timespec="seconds")},
    {"item": "python", "value": sys.version.replace("\n", " ")},
    {"item": "python_executable", "value": sys.executable},
    {"item": "platform", "value": platform.platform()},
    {"item": "project_root", "value": str(ROOT)},
    {"item": "virtual_env", "value": os.environ.get("VIRTUAL_ENV", "not set")},
    {"item": "git_branch", "value": git_branch if git_code == 0 else "unknown"},
    {"item": "git_dirty_entries", "value": len(git_status.splitlines()) if git_status_code == 0 else "unknown"},
]

runtime_df = pd.DataFrame(runtime_rows)
display(runtime_df)

,item,value
0,timestamp_local,2026-04-29T09:20:49
1,python,"3.11.15 (main, Mar 3 2026, 09:26:23) [GCC 13...."
2,python_executable,/home/endy/menuforge/.venv/bin/python
3,platform,Linux-5.15.167.4-microsoft-standard-WSL2-x86_6...
4,project_root,/home/endy/menuforge
5,virtual_env,/home/endy/menuforge/.venv
6,git_branch,main
7,git_dirty_entries,26


In [4]:
hardware_rows = []

try:
    import torch

    cuda_available = bool(torch.cuda.is_available())
    hardware_rows.append({"item": "torch_version", "value": torch.__version__, "status": "ok"})
    hardware_rows.append({"item": "torch_cuda_available", "value": cuda_available, "status": "ok"})
    if cuda_available:
        device_index = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(device_index)
        hardware_rows.extend(
            [
                {"item": "cuda_device_name", "value": props.name, "status": "ok"},
                {"item": "cuda_total_vram", "value": bytes_human(props.total_memory), "status": "ok"},
                {"item": "cuda_device_count", "value": torch.cuda.device_count(), "status": "ok"},
            ]
        )
except Exception as exc:
    hardware_rows.append({"item": "torch_cuda_check", "value": repr(exc), "status": "warning"})

code, smi_out, smi_err = run_cmd(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
    timeout=15,
)
hardware_rows.append(
    {
        "item": "nvidia_smi",
        "value": smi_out if code == 0 else smi_err or "not available",
        "status": "ok" if code == 0 else "warning",
    }
)

meminfo = {}
meminfo_path = Path("/proc/meminfo")
if meminfo_path.exists():
    for line in meminfo_path.read_text().splitlines():
        key, value = line.split(":", 1)
        meminfo[key] = int(value.strip().split()[0]) * 1024
hardware_rows.append(
    {"item": "system_ram_total", "value": bytes_human(meminfo.get("MemTotal")), "status": "ok" if meminfo else "warning"}
)

for label, path in [("project_disk", ROOT), ("tmp_disk", Path("/tmp"))]:
    usage = shutil.disk_usage(path)
    hardware_rows.append(
        {
            "item": label,
            "value": f"free {bytes_human(usage.free)} / total {bytes_human(usage.total)}",
            "status": "ok",
        }
    )

hardware_df = pd.DataFrame(hardware_rows)
display(hardware_df)

,item,value,status
0,torch_version,2.11.0+cu130,ok
1,torch_cuda_available,True,ok
2,cuda_device_name,NVIDIA GeForce RTX 4060 Laptop GPU,ok
3,cuda_total_vram,8.0 GB,ok
4,cuda_device_count,1,ok
5,nvidia_smi,"NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB, ...",ok
6,system_ram_total,58.9 GB,ok
7,project_disk,free 796.4 GB / total 1006.9 GB,ok
8,tmp_disk,free 796.4 GB / total 1006.9 GB,ok


## Package Audit

In [5]:
package_specs = [
    {"dist": "pandas", "module": "pandas", "role": "tabular manifests"},
    {"dist": "numpy", "module": "numpy", "role": "arrays and metrics"},
    {"dist": "pyarrow", "module": "pyarrow", "role": "parquet output"},
    {"dist": "duckdb", "module": "duckdb", "role": "local tabular queries"},
    {"dist": "datasets", "module": "datasets", "role": "Hugging Face dataset loading"},
    {"dist": "huggingface-hub", "module": "huggingface_hub", "role": "model/dataset downloads"},
    {"dist": "transformers", "module": "transformers", "role": "HF model loading"},
    {"dist": "torch", "module": "torch", "role": "GPU checks and model runtime"},
    {"dist": "torchvision", "module": "torchvision", "role": "vision transforms"},
    {"dist": "accelerate", "module": "accelerate", "role": "large model/offload support"},
    {"dist": "bitsandbytes", "module": "bitsandbytes", "role": "quantized training/inference support"},
    {"dist": "Pillow", "module": "PIL", "role": "image IO"},
    {"dist": "opencv-contrib-python", "module": "cv2", "role": "image preprocessing"},
    {"dist": "rapidfuzz", "module": "rapidfuzz", "role": "text matching metrics"},
    {"dist": "jiwer", "module": "jiwer", "role": "OCR WER/CER metrics"},
    {"dist": "lancedb", "module": "lancedb", "role": "future retrieval continuity"},
    {"dist": "paddleocr", "module": "paddleocr", "role": "OCR baseline package"},
    {"dist": "paddlepaddle", "module": "paddle", "role": "PaddleOCR runtime"},
    {"dist": "diffusers", "module": "diffusers", "role": "image-generation probes"},
]

package_rows = []
for spec in package_specs:
    version = package_version(spec["dist"])
    if version:
        try:
            importlib.import_module(spec["module"])
            import_status = "import_ok"
        except Exception as exc:
            import_status = f"import_failed: {type(exc).__name__}"
    else:
        import_status = "not_checked"
    package_rows.append(
        {
            "package": spec["dist"],
            "module": spec["module"],
            "version": version or "not installed",
            "role": spec["role"],
            "status": "installed" if version else "missing_or_later_phase",
            "import_status": import_status,
        }
    )

packages_df = pd.DataFrame(package_rows)
display(packages_df)

/home/endy/menuforge/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


,package,module,version,role,status,import_status
0,pandas,pandas,2.3.3,tabular manifests,installed,import_ok
1,numpy,numpy,2.3.5,arrays and metrics,installed,import_ok
2,pyarrow,pyarrow,24.0.0,parquet output,installed,import_ok
3,duckdb,duckdb,1.5.2,local tabular queries,installed,import_ok
4,datasets,datasets,4.8.4,Hugging Face dataset loading,installed,import_ok
5,huggingface-hub,huggingface_hub,0.36.2,model/dataset downloads,installed,import_ok
6,transformers,transformers,4.57.6,HF model loading,installed,import_ok
7,torch,torch,2.11.0,GPU checks and model runtime,installed,import_ok
8,torchvision,torchvision,0.26.0,vision transforms,installed,import_ok
9,accelerate,accelerate,1.13.0,large model/offload support,installed,import_ok


## Local Model Inventory

In [6]:
expected_models = [
    {"id": "model-ocrvl", "name": "PaddleOCR-VL", "path": "models/paddleocr-vl", "phase": "SBV2_05", "key_files": ["config.json", "model.safetensors", "PP-DocLayoutV2/inference.pdiparams"]},
    {"id": "model-qwen-vl7b", "name": "Qwen2.5-VL-7B GGUF", "path": "models/Qwen_Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf", "phase": "SBV2_05", "key_files": []},
    {"id": "model-qwen-vl7b-mmproj", "name": "Qwen2.5-VL mmproj", "path": "models/mmproj-Qwen_Qwen2.5-VL-7B-Instruct-f16.gguf", "phase": "SBV2_05", "key_files": []},
    {"id": "model-layoutlmv3", "name": "LayoutLMv3 Base", "path": "models/layoutlmv3-base", "phase": "SBV2_05", "key_files": ["config.json", "model.safetensors"]},
    {"id": "model-clip", "name": "CLIP ViT-B/32", "path": "models/clip-vit-base-patch32", "phase": "optional", "key_files": ["config.json", "pytorch_model.bin"]},
    {"id": "model-dinov2", "name": "DINOv2 Base", "path": "models/dinov2-base", "phase": "optional", "key_files": ["config.json", "model.safetensors"]},
    {"id": "model-vjepa2", "name": "V-JEPA 2 local sidecar", "path": "models/vjepa2-vitl-fpc64-256", "phase": "optional", "key_files": ["config.json", "model.safetensors"]},
    {"id": "model-zimage-turbo", "name": "Z-Image-Turbo", "path": "models/z-image-turbo", "phase": "SBV2_11", "key_files": ["model_index.json", "transformer/diffusion_pytorch_model.safetensors.index.json"]},
    {"id": "model-qwen-edit-2509", "name": "Qwen-Image-Edit-2509", "path": "models/qwen-image-edit-2509", "phase": "SBV2_11", "key_files": ["model_index.json", "transformer/diffusion_pytorch_model.safetensors.index.json"]},
    {"id": "model-sd35-medium", "name": "Stable Diffusion 3.5 Medium", "path": "models/sd3.5-medium", "phase": "SBV2_11", "key_files": ["model_index.json"]},
]

inventory_rows = []
for spec in expected_models:
    local_path = ROOT / spec["path"]
    incomplete_files = list(local_path.rglob("*.incomplete")) if local_path.exists() and local_path.is_dir() else []
    missing_key_files = [name for name in spec["key_files"] if not (local_path / name).exists()]
    if not local_path.exists():
        status = "missing"
    elif incomplete_files:
        status = "partial_incomplete_files"
    elif local_path.name == "sd3.5-medium" and missing_key_files:
        status = "metadata_only_gated_skip"
    elif local_path.is_dir() and missing_key_files:
        status = "present_but_check_key_files"
    else:
        status = "ready"

    inventory_rows.append(
        {
            "id": spec["id"],
            "name": spec["name"],
            "phase": spec["phase"],
            "local_path": spec["path"],
            "exists": local_path.exists(),
            "status": status,
            "size": bytes_human(du_bytes(local_path)),
            "file_count": count_files(local_path) if local_path.is_dir() else (1 if local_path.exists() else 0),
            "incomplete_count": len(incomplete_files),
            "missing_key_files": ", ".join(missing_key_files),
        }
    )

model_inventory_df = pd.DataFrame(inventory_rows)
display(model_inventory_df)

,id,name,phase,local_path,exists,status,size,file_count,incomplete_count,missing_key_files
0,model-ocrvl,PaddleOCR-VL,SBV2_05,models/paddleocr-vl,True,ready,2.0 GB,47,0,
1,model-qwen-vl7b,Qwen2.5-VL-7B GGUF,SBV2_05,models/Qwen_Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf,True,ready,4.4 GB,1,0,
2,model-qwen-vl7b-mmproj,Qwen2.5-VL mmproj,SBV2_05,models/mmproj-Qwen_Qwen2.5-VL-7B-Instruct-f16....,True,ready,1.3 GB,1,0,
3,model-layoutlmv3,LayoutLMv3 Base,SBV2_05,models/layoutlmv3-base,True,ready,1.9 GB,23,0,
4,model-clip,CLIP ViT-B/32,optional,models/clip-vit-base-patch32,True,ready,1.7 GB,25,0,
5,model-dinov2,DINOv2 Base,optional,models/dinov2-base,True,ready,660.7 MB,13,0,
6,model-vjepa2,V-JEPA 2 local sidecar,optional,models/vjepa2-vitl-fpc64-256,True,ready,6.0 GB,17,0,
7,model-zimage-turbo,Z-Image-Turbo,SBV2_11,models/z-image-turbo,True,ready,30.6 GB,65,0,
8,model-qwen-edit-2509,Qwen-Image-Edit-2509,SBV2_11,models/qwen-image-edit-2509,True,ready,53.8 GB,71,0,
9,model-sd35-medium,Stable Diffusion 3.5 Medium,SBV2_11,models/sd3.5-medium,True,metadata_only_gated_skip,23.5 KB,5,0,model_index.json


## Source And Model Register Preview

In [7]:
def parse_markdown_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = []
    headers = None
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip().startswith("|"):
            continue
        parts = [part.strip() for part in line.strip().strip("|").split("|")]
        if all(re.fullmatch(r":?-{3,}:?", part) for part in parts):
            continue
        if headers is None:
            headers = parts
            continue
        if len(parts) == len(headers):
            rows.append(dict(zip(headers, parts)))
    return pd.DataFrame(rows)


register_df = parse_markdown_table(REGISTER_PATH)
display(register_df)

if register_df.empty:
    print("Register table not found or empty.")
else:
    print(f"Register rows: {len(register_df)}")

,ID,Type,Name,Version/date,Source URL,Local path,Licence/terms,Intended use,Local plan,Status
0,data-nypl,dataset,NYPL What's on the Menu,existing local export,https://www.nypl.org/research/support/whats-on...,`data/raw/nypl/`,verify per NYPL source,real menu page and dish text sample,local sample and preview,pending sample
1,data-cord,dataset,CORD receipts,v2 smoke verified 2026-04-29,https://github.com/clovaai/cord,`data/raw/sandbox_v2/smoke_sources/cord/`; ful...,CC BY 4.0 per project repo,optional item/price/quantity structure stress ...,defer for English-first pass; Indonesian sourc...,smoke verified; future/OOD
2,data-sroie,dataset,ICDAR2019 SROIE,HF smoke verified 2026-04-29,https://huggingface.co/datasets/jsdnrs/ICDAR20...,`data/raw/sandbox_v2/smoke_sources/sroie/`; fu...,CC BY 4.0 per HF card,English OCR and key information extraction,first external download/sample,smoke verified
3,data-funsd,dataset,FUNSD,HF smoke verified 2026-04-29,https://huggingface.co/datasets/nielsr/funsd,`data/raw/sandbox_v2/smoke_sources/funsd/`; fu...,verify exact dataset licence before reported r...,English noisy scanned form layout/entity extra...,first external download/sample,smoke verified
4,data-korean-menu,dataset,Korean Menus Image Dataset,HF card,https://huggingface.co/datasets/HumynLabs/Kore...,`data/raw/sandbox_v2/korean_menus/`,CC BY 4.0 per HF card,future multilingual/OOD menu sanity check,defer; not part of English-first benchmark,future/OOD
5,data-doclaynet,dataset,DocLayNet sample,optional,https://research.ibm.com/publications/doclayne...,`data/raw/sandbox_v2/doclaynet_sample/`,verify before use,optional layout-diversity stress test,postpone unless needed,optional
6,model-ocr,model/tool,PaddleOCR + PaddlePaddle CPU runtime,PaddleOCR 3.5.0 / PaddlePaddle 3.2.0 verified ...,https://github.com/PaddlePaddle/PaddleOCR,`.venv/`; PaddleX cache via `PADDLE_PDX_CACHE_...,Apache 2.0 for PaddleOCR/PaddleX/PaddlePaddle ...,OCR baseline,first local extraction model; use `sandbox_v2/...,import verified
7,model-ocrvl,model,PaddleOCR-VL-0.9B,exact checkpoint TBD,https://huggingface.co/PaddlePaddle/PaddleOCR-VL,`models/` or HF cache,verify exact checkpoint,compact document VLM extraction,preferred local VLM first ride,pending
8,model-qwen-vl7b,model,Qwen2.5-VL-7B-Instruct,exact checkpoint or existing local route,https://huggingface.co/Qwen/Qwen2.5-VL-7B-Inst...,"`models/`, HF cache or existing local runtime",Apache 2.0 per HF card,richer VLM extraction and critique,local only if quantized/already working,pending
9,model-qwen-vl3b,model,Qwen2.5-VL-3B-Instruct,optional,https://huggingface.co/Qwen/Qwen2.5-VL-3B-Inst...,`models/` or HF cache,verify exact licence before use,lighter VLM fallback,research fallback only if licence acceptable,optional


Register rows: 18


In [8]:
if register_df.empty or "ID" not in register_df.columns:
    register_model_check_df = pd.DataFrame()
else:
    model_rows = register_df[register_df["Type"].str.contains("model", case=False, na=False)].copy()
    register_model_check_df = model_rows.merge(
        model_inventory_df[["id", "status", "size", "exists", "incomplete_count"]],
        left_on="ID",
        right_on="id",
        how="left",
    )
    inventory_status_col = "status_y" if "status_y" in register_model_check_df.columns else "status"
    register_model_check_df["local_inventory_status"] = register_model_check_df[inventory_status_col].fillna("not_in_inventory_list")
    register_model_check_df = register_model_check_df.drop(columns=[col for col in ["id", inventory_status_col] if col in register_model_check_df.columns])

display(register_model_check_df)

,ID,Type,Name,Version/date,Source URL,Local path,Licence/terms,Intended use,Local plan,Status,size,exists,incomplete_count,local_inventory_status
0,model-ocr,model/tool,PaddleOCR + PaddlePaddle CPU runtime,PaddleOCR 3.5.0 / PaddlePaddle 3.2.0 verified ...,https://github.com/PaddlePaddle/PaddleOCR,`.venv/`; PaddleX cache via `PADDLE_PDX_CACHE_...,Apache 2.0 for PaddleOCR/PaddleX/PaddlePaddle ...,OCR baseline,first local extraction model; use `sandbox_v2/...,import verified,NaN,NaN,NaN,not_in_inventory_list
1,model-ocrvl,model,PaddleOCR-VL-0.9B,exact checkpoint TBD,https://huggingface.co/PaddlePaddle/PaddleOCR-VL,`models/` or HF cache,verify exact checkpoint,compact document VLM extraction,preferred local VLM first ride,pending,2.0 GB,True,0.0,ready
2,model-qwen-vl7b,model,Qwen2.5-VL-7B-Instruct,exact checkpoint or existing local route,https://huggingface.co/Qwen/Qwen2.5-VL-7B-Inst...,"`models/`, HF cache or existing local runtime",Apache 2.0 per HF card,richer VLM extraction and critique,local only if quantized/already working,pending,4.4 GB,True,0.0,ready
3,model-qwen-vl3b,model,Qwen2.5-VL-3B-Instruct,optional,https://huggingface.co/Qwen/Qwen2.5-VL-3B-Inst...,`models/` or HF cache,verify exact licence before use,lighter VLM fallback,research fallback only if licence acceptable,optional,NaN,NaN,NaN,not_in_inventory_list
4,model-layoutlmv3,model,LayoutLMv3,exact checkpoint TBD,https://www.microsoft.com/en-us/research/publi...,`models/` or HF cache,verify exact checkpoint,OCR+layout document baseline,local CPU/GPU after OCR boxes exist,pending,1.9 GB,True,0.0,ready
5,model-clip,model,CLIP baseline,exact checkpoint TBD,https://proceedings.mlr.press/v139/radford21a....,`models/` or HF cache,verify exact checkpoint,image-text alignment baseline,"local, optional after extraction",optional,1.7 GB,True,0.0,ready
6,model-dinov2,model,DINOv2,exact checkpoint TBD,https://arxiv.org/abs/2304.07193,`models/` or HF cache,verify exact checkpoint,visual clustering/outlier baseline,"local, optional after extraction",optional,660.7 MB,True,0.0,ready
7,model-qwen-image,model,Qwen-Image-2512,exact checkpoint,https://huggingface.co/Qwen/Qwen-Image-2512,later cloud GPU cache,Apache 2.0 per HF card,controlled one-shot menu generation,download metadata only for now; do not require...,later,NaN,NaN,NaN,not_in_inventory_list
8,model-zimage-turbo,model,Z-Image-Turbo,HF card verified 2026-04-29,https://huggingface.co/Tongyi-MAI/Z-Image-Turbo,`models/z-image-turbo/` or external training c...,Apache 2.0 per HF card,first low-VRAM local adapter/generation probe,"try 512px, batch 1, LoRA rank 4-8, quantisatio...",planned,30.6 GB,True,0.0,ready
9,model-qwen-edit-2509,model,Qwen-Image-Edit-2509,HF card verified 2026-04-29,https://huggingface.co/Qwen/Qwen-Image-Edit-2509,`models/qwen-image-edit-2509/` or external tra...,Apache 2.0 per HF card,Qwen-family low-VRAM edit/adaptation probe,try after Z-Image-Turbo with AI Toolkit-style ...,planned,53.8 GB,True,0.0,ready


## Persist Audit Tables

In [9]:
runtime_df.to_csv(AUDIT_DIR / "sbv2_00_runtime_summary.csv", index=False)
hardware_df.to_csv(AUDIT_DIR / "sbv2_00_hardware_summary.csv", index=False)
packages_df.to_csv(AUDIT_DIR / "sbv2_00_package_audit.csv", index=False)
model_inventory_df.to_csv(AUDIT_DIR / "sbv2_00_model_inventory.csv", index=False)
register_df.to_csv(AUDIT_DIR / "sbv2_00_register_preview.csv", index=False)
register_model_check_df.to_csv(AUDIT_DIR / "sbv2_00_register_model_check.csv", index=False)

summary = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(ROOT),
    "model_status_counts": model_inventory_df["status"].value_counts().to_dict(),
    "missing_packages": packages_df.loc[packages_df["status"] != "installed", "package"].tolist(),
    "import_failures": packages_df.loc[(packages_df["status"] == "installed") & (packages_df["import_status"] != "import_ok"), "package"].tolist(),
}
(AUDIT_DIR / "sbv2_00_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Wrote audit files:")
for path in sorted(AUDIT_DIR.glob("sbv2_00_*")):
    print(f"- {path.relative_to(ROOT)}")

Wrote audit files:
- outputs/sandbox_v2/audit_tables/sbv2_00_hardware_summary.csv
- outputs/sandbox_v2/audit_tables/sbv2_00_model_inventory.csv
- outputs/sandbox_v2/audit_tables/sbv2_00_package_audit.csv
- outputs/sandbox_v2/audit_tables/sbv2_00_register_model_check.csv
- outputs/sandbox_v2/audit_tables/sbv2_00_register_preview.csv
- outputs/sandbox_v2/audit_tables/sbv2_00_runtime_summary.csv
- outputs/sandbox_v2/audit_tables/sbv2_00_summary.json


## Readiness Notes Before SBV2 01

Before moving to `SBV2_01_data_download_and_manifest.ipynb`, review:

- whether CUDA appears correctly in the hardware table
- whether `paddleocr`, `paddlepaddle` or `diffusers` are missing or have non-OK import status
- whether any expected model has `partial_incomplete_files`
- whether SD3.5 remains `metadata_only_gated_skip`, which is acceptable for now
- whether the source/model register needs status updates after the audit